# Analyse complète des motifs de rejet

Ce notebook :
1. charge les données de rejet ;
2. extrait les événements de rejet depuis l'historique ;
3. sépare les motifs en une ligne par motif ;
4. nettoie les textes sans perdre les actions importantes ;
5. explore les n-grammes ;
6. regroupe les motifs similaires ;
7. attribue des thèmes métier ;
8. génère un rapport Excel complet.

In [ ]:
# À lancer uniquement si les bibliothèques ne sont pas déjà installées.
%pip install pandas openpyxl matplotlib seaborn scikit-learn

In [ ]:
# Cette cellule rassemble les bibliothèques et tous les paramètres modifiables.

from pathlib import Path
from collections import Counter
from io import BytesIO
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

from openpyxl import load_workbook
from openpyxl.drawing.image import Image as ExcelImage

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 250)
sns.set_theme(style="whitegrid", context="notebook")

# -----------------------------
# Fichiers d'entrée et de sortie
# -----------------------------
DATA_PATH = Path("data/query_result.xlsx")
SHEET_NAME = "Résultats de la requête"

OUTPUT_DIR = Path("data/output")
FIGURES_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

EVENTS_PATH = OUTPUT_DIR / "rejection_events.csv"
MOTIFS_PATH = OUTPUT_DIR / "rejection_reasons_detailed.csv"
REPORT_PATH = OUTPUT_DIR / "rapport_analyse_motifs_rejet.xlsx"

# -----------------------------
# Colonnes attendues dans Excel
# -----------------------------
TARGET_COL = "Historique Rejets"
REGION_COL = "Region"
OPERATOR_COL = "Nom Operateur Court"

REQUIRED_COLUMNS = [
    TARGET_COL,
    REGION_COL,
    OPERATOR_COL,
    "Niveau",
    "Nb Occurrences Rejet",
    "Nb Dossiers Distincts",
]

# -----------------------------------------
# Paramètres de représentation par n-grammes
# -----------------------------------------
NGRAM_RANGE = (1, 3)   # mots seuls, bigrammes et trigrammes
MIN_DF = 3             # un n-gramme doit apparaître au moins 3 fois
MAX_DF = 0.90          # exclut les expressions présentes dans presque tous les motifs
N_CLUSTERS = 20        # nombre initial de groupes automatiques à examiner

DATE_FORMAT = "%d/%m/%Y %H:%M"

In [ ]:
# Fonctions de normalisation générale.
# Elles sont utilisées à plusieurs étapes du notebook.

def strip_accents(text):
    """Supprime les accents pour rapprocher les variantes : signé / signe."""
    if pd.isna(text):
        return ""
    normalized = unicodedata.normalize("NFKD", str(text))
    return "".join(char for char in normalized if not unicodedata.combining(char))


def normalize_spaces(text):
    """Remplace les espaces, tabulations et retours ligne multiples par un espace."""
    if pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()


def safe_sheet_name(name):
    """Garantit un nom de feuille Excel valide et inférieur à 31 caractères."""
    return re.sub(r'[:\\/*?\[\]]', "_", str(name))[:31]

In [ ]:
# Cette étape charge le fichier Excel et vérifie que les données nécessaires
# sont disponibles avant de commencer les transformations.

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {DATA_PATH}\n"
        "Placez le fichier source dans le dossier data/."
    )

df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
df.columns = df.columns.map(str).str.strip()

missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
if missing_columns:
    raise ValueError(
        "Colonnes obligatoires absentes du fichier source : "
        + ", ".join(missing_columns)
    )

# Harmonisation des colonnes texte.
for col in [TARGET_COL, "Niveau", REGION_COL, OPERATOR_COL]:
    df[col] = df[col].astype("string").str.strip()

# Harmonisation des colonnes numériques.
for col in ["Nb Occurrences Rejet", "Nb Dossiers Distincts"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["source_row_id"] = df.index

quality_data = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean() * 100).round(2).values,
    "unique_values": df.nunique(dropna=True).values,
})

print(f"Nombre de lignes source : {len(df):,}")
print(f"Nombre de colonnes source : {df.shape[1]:,}")
print(f"Doublons complets : {df.duplicated().sum():,}")

display(df.head())
display(quality_data.sort_values("missing_pct", ascending=False))
display(
    df[["Nb Occurrences Rejet", "Nb Dossiers Distincts"]]
    .describe()
    .T
)

In [ ]:
# Un historique peut contenir plusieurs événements.
# Cette étape crée une ligne par événement de rejet tout en gardant les
# informations de contexte : région, opérateur, niveau et compteurs source.

EVENT_RE = re.compile(
    r"(?ms)"
    r"#(?P<event_num>\d+)\s*"
    r"\[(?P<event_level>[^\]]+)\]:\s*"
    r"(?P<rejected_at>\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2})\s*"
    r"\|\s*"
    r"(?P<body>.*?)(?=\n#\d+\s*\[[^\]]+\]:|\Z)"
)

CORRECTED_RE = re.compile(
    r"→\s*Corrigé:\s*"
    r"(?P<corrected_at>\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2})"
    r"(?:\s*\[Écart:\s*(?P<gap>[^\]]+)\])?",
    flags=re.IGNORECASE,
)

ACTOR_RE = re.compile(
    r"\((?P<actor>[^()]{3,120})\)\s*→\s*Corrigé",
    flags=re.IGNORECASE,
)


def parse_gap_to_hours(gap):
    """Transforme un délai de type '2j 4h 30m' en nombre d'heures."""
    if pd.isna(gap) or not str(gap).strip():
        return pd.NA

    text = str(gap).lower()
    days = re.search(r"(\d+)\s*j", text)
    hours = re.search(r"(\d+)\s*h", text)
    minutes = re.search(r"(\d+)\s*m", text)

    total = 0.0
    if days:
        total += int(days.group(1)) * 24
    if hours:
        total += int(hours.group(1))
    if minutes:
        total += int(minutes.group(1)) / 60

    return total if total > 0 else pd.NA


def parse_history(history, row):
    """Extrait tous les événements contenus dans un historique de rejet."""
    if pd.isna(history) or not str(history).strip():
        return []

    text = str(history).replace("\r\n", "\n")
    matches = list(EVENT_RE.finditer(text))

    # Si le format attendu n'est pas trouvé, l'historique est gardé comme
    # un événement unique pour éviter de perdre l'information.
    if not matches:
        matches = [None]

    events = []

    for event_index, match in enumerate(matches, start=1):
        if match is None:
            body = text
            event_num = pd.NA
            event_level = pd.NA
            rejected_at = pd.NaT
        else:
            body = match.group("body").strip()
            event_num = int(match.group("event_num"))
            event_level = match.group("event_level").strip()
            rejected_at = pd.to_datetime(
                match.group("rejected_at"),
                format=DATE_FORMAT,
                errors="coerce",
            )

        corrected_match = CORRECTED_RE.search(body)
        actor_match = ACTOR_RE.search(body)

        actor = actor_match.group("actor").strip() if actor_match else pd.NA
        corrected_at = (
            pd.to_datetime(
                corrected_match.group("corrected_at"),
                format=DATE_FORMAT,
                errors="coerce",
            )
            if corrected_match
            else pd.NaT
        )
        gap_text = (
            corrected_match.group("gap")
            if corrected_match and corrected_match.group("gap")
            else pd.NA
        )

        events.append({
            "source_row_id": row["source_row_id"],
            "event_index": event_index,
            "event_number": event_num,
            "event_level": event_level,
            "rejected_at": rejected_at,
            "corrected_at": corrected_at,
            "gap_text": gap_text,
            "gap_hours": parse_gap_to_hours(gap_text),
            "actor": actor,
            "raw_event": body,
            "Niveau": row.get("Niveau"),
            REGION_COL: row.get(REGION_COL),
            OPERATOR_COL: row.get(OPERATOR_COL),
            "Nb Occurrences Rejet": row.get("Nb Occurrences Rejet"),
            "Nb Dossiers Distincts": row.get("Nb Dossiers Distincts"),
        })

    return events


event_rows = [
    event
    for _, row in df.iterrows()
    for event in parse_history(row[TARGET_COL], row)
]

events = pd.DataFrame(event_rows)

if events.empty:
    raise ValueError("Aucun événement de rejet n'a pu être extrait.")

events = events.reset_index(names="event_id")
events["gap_hours"] = pd.to_numeric(events["gap_hours"], errors="coerce")
events["gap_days"] = events["gap_hours"] / 24
events["is_corrected"] = events["corrected_at"].notna()

print(f"Événements extraits : {len(events):,}")
display(events.head(10))

In [ ]:
# Un événement peut décrire plusieurs motifs de rejet.
# Cette étape isole les puces, paragraphes et phrases afin d'obtenir
# une ligne par motif exploitable.

SECTION_LINE_RE = re.compile(r"(?m)^\s*[^\n:]{3,80}:\s*$")
BULLET_LINE_RE = re.compile(r"(?m)^\s*[-•*]\s*(.+?)\s*$")


def remove_correction_metadata(text, actor=pd.NA):
    """Supprime la date de correction et le nom de l'acteur du texte à analyser."""
    text = str(text).replace("\r\n", "\n")
    text = CORRECTED_RE.sub("", text)

    if not pd.isna(actor):
        actor_clean = " ".join(str(actor).split())
        text = text.replace(f"({actor})", "")
        text = re.sub(
            r"\(\s*" + re.escape(actor_clean) + r"\s*\)",
            "",
            text,
        )

    return text


def normalize_reason(reason):
    """Nettoyage léger avant la phase de nettoyage détaillé."""
    return normalize_spaces(str(reason).strip(" -|:;,. \n\r\t"))


def split_reasons_from_event(row):
    """Découpe un événement en plusieurs motifs candidats."""
    text = remove_correction_metadata(row["raw_event"], row["actor"])

    bullet_reasons = [
        normalize_reason(reason)
        for reason in BULLET_LINE_RE.findall(text)
    ]

    remainder = BULLET_LINE_RE.sub("\n", text)
    remainder = SECTION_LINE_RE.sub("\n", remainder)

    candidates = [
        normalize_reason(chunk)
        for chunk in re.split(r"\n+|\s+-\s+|(?<=[.!?])\s+", remainder)
    ]

    reasons = []
    seen = set()

    for reason in bullet_reasons + candidates:
        normalized_key = strip_accents(reason).lower()

        # Un motif très court est généralement un séparateur ou une information
        # inutilisable. Le texte original reste disponible dans raw_event.
        if len(normalized_key) < 8:
            continue

        if normalized_key not in seen:
            seen.add(normalized_key)
            reasons.append(reason)

    return reasons


reason_rows = []

for _, event in events.iterrows():
    split_reasons = split_reasons_from_event(event)

    for reason_index, reason_raw in enumerate(split_reasons, start=1):
        reason_rows.append({
            "event_id": event["event_id"],
            "source_row_id": event["source_row_id"],
            "reason_index": reason_index,
            "reason_raw": reason_raw,
            "event_level": event["event_level"],
            "rejected_at": event["rejected_at"],
            "corrected_at": event["corrected_at"],
            "gap_days": event["gap_days"],
            REGION_COL: event[REGION_COL],
            OPERATOR_COL: event[OPERATOR_COL],
        })

motifs = pd.DataFrame(reason_rows).reset_index(names="reason_id")

if motifs.empty:
    raise ValueError("Aucun motif atomique n'a pu être créé.")

print(f"Motifs atomiques extraits : {len(motifs):,}")
display(motifs.head(20))

In [ ]:
# Cette cellule ne modifie jamais reason_raw.
#
# reason_clean : version lisible et normalisée ;
# reason_model : version utilisée par les n-grammes et le clustering.
#
# Les actions importantes sont conservées et standardisées.
# Exemple : "à compléter" devient "information_a_completer".
# Ainsi, le sens métier n'est pas perdu.

GENERIC_POLITENESS_PATTERNS = [
    r"\bveuillez\b",
    r"\bmerci de\b",
    r"\bpriere de\b",
    r"\bnous vous prions de\b",
    r"\bil faut\b",
]

# Ces normalisations rapprochent les formulations équivalentes.
# Elles conservent volontairement l'action demandée.
ACTION_NORMALIZATIONS = {
    r"\b(a|à)\s+completer\b": "information_a_completer",
    r"\ba\s+renseigner\b": "information_a_renseigner",
    r"\ba\s+preciser\b": "information_a_preciser",
    r"\ba\s+fournir\b": "document_a_fournir",
    r"\ba\s+joindre\b": "document_a_joindre",
    r"\ba\s+signer\b": "signature_a_fournir",
    r"\bnon\s+signe\b": "signature_non_signee",
    r"\bpas\s+signe\b": "signature_non_signee",
    r"\babsence\s+de\s+signature\b": "signature_absente",
    r"\bchamps?\s+vides?\b": "champ_non_renseigne",
    r"\bnon\s+renseigne\b": "champ_non_renseigne",
    r"\binformation\s+manquante\b": "information_manquante",
    r"\bpieces?\s+manquantes?\b": "piece_manquante",
    r"\bshap\s*file\b": "shapefile",
    r"\bp\.?\s*v\.?\s*c\.?\s*l\.?\b": "pvcl",
}

# Mots très génériques sans valeur forte pour le regroupement.
# Les actions importantes ne sont pas dans cette liste.
STOPWORDS_FR = {
    "le", "la", "les", "un", "une", "des", "du", "de", "d", "et", "ou",
    "en", "dans", "sur", "pour", "par", "avec", "sans", "au", "aux",
    "ce", "cet", "cette", "ces", "son", "sa", "ses", "leur", "leurs",
    "qui", "que", "quoi", "dont", "est", "sont", "etre", "avoir",
    "fait", "faire", "ainsi", "plus", "moins", "tres", "bien",
    "merci", "veuillez", "priere", "corrige", "corriger", "correction",
    "rejet", "rejete", "dossier", "niveau",
}


def clean_reason_text(text):
    """Crée une version normalisée lisible du motif."""
    if pd.isna(text):
        return ""

    cleaned = str(text).lower()
    cleaned = strip_accents(cleaned)
    cleaned = cleaned.replace("\r\n", " ").replace("\n", " ").replace("\t", " ")

    # Les formulations de politesse disparaissent, mais l'action est préservée.
    for pattern in GENERIC_POLITENESS_PATTERNS:
        cleaned = re.sub(pattern, " ", cleaned)

    # Les actions et expressions métier sont normalisées avant suppression
    # des caractères spéciaux ou des mots vides.
    for pattern, replacement in ACTION_NORMALIZATIONS.items():
        cleaned = re.sub(pattern, replacement, cleaned)

    # Les underscores sont conservés car ils portent les expressions métier
    # normalisées : information_a_completer, signature_non_signee, etc.
    cleaned = re.sub(r"[^a-z0-9_'\s]+", " ", cleaned)
    cleaned = normalize_spaces(cleaned)

    return cleaned


def build_model_text(clean_text):
    """Prépare le texte spécifiquement pour les n-grammes."""
    tokens = re.findall(r"\b[a-z][a-z0-9_']{1,}\b", clean_text)

    tokens = [
        token
        for token in tokens
        if token not in STOPWORDS_FR
        and len(token) >= 2
    ]

    return " ".join(tokens)


motifs["reason_clean"] = motifs["reason_raw"].map(clean_reason_text)
motifs["reason_model"] = motifs["reason_clean"].map(build_model_text)
motifs["reason_word_count"] = motifs["reason_model"].str.split().str.len()
motifs["is_valid_reason"] = motifs["reason_model"].str.len().fillna(0).ge(3)

print(f"Motifs valides : {motifs['is_valid_reason'].sum():,}")
print(f"Motifs vides ou trop courts : {(~motifs['is_valid_reason']).sum():,}")

display(
    motifs[
        ["reason_raw", "reason_clean", "reason_model", "is_valid_reason"]
    ].head(20)
)

In [ ]:
# Cette étape permet de comprendre les expressions dominantes avant
# de former des groupes automatiques.

valid_texts = motifs.loc[motifs["is_valid_reason"], "reason_model"]

ngram_exploration = TfidfVectorizer(
    ngram_range=NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    use_idf=False,
    norm=None,
)

try:
    ngram_matrix = ngram_exploration.fit_transform(valid_texts)
except ValueError:
    # Solution de secours si le jeu de données est très petit.
    ngram_exploration = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=1,
        use_idf=False,
        norm=None,
    )
    ngram_matrix = ngram_exploration.fit_transform(valid_texts)

ngram_counts = pd.DataFrame({
    "ngram": ngram_exploration.get_feature_names_out(),
    "count": np.asarray(ngram_matrix.sum(axis=0)).ravel(),
}).sort_values("count", ascending=False)

unigrams = ngram_counts[
    ngram_counts["ngram"].str.split().str.len().eq(1)
].head(30)

bigrams = ngram_counts[
    ngram_counts["ngram"].str.split().str.len().eq(2)
].head(30)

trigrams = ngram_counts[
    ngram_counts["ngram"].str.split().str.len().eq(3)
].head(30)

display(unigrams)
display(bigrams)
display(trigrams)

In [ ]:
# Les motifs sont représentés par des vecteurs TF-IDF de n-grammes.
# Les motifs qui partagent des expressions proches sont regroupés dans
# un même cluster. Ces clusters servent ensuite à guider la revue métier.

cluster_vectorizer = TfidfVectorizer(
    ngram_range=NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    sublinear_tf=True,
)

try:
    X = cluster_vectorizer.fit_transform(valid_texts)
except ValueError:
    cluster_vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
    )
    X = cluster_vectorizer.fit_transform(valid_texts)

valid_indices = motifs.index[motifs["is_valid_reason"]].to_numpy()
non_zero_mask = np.asarray(X.sum(axis=1)).ravel() > 0

motifs["cluster_id"] = -1

if non_zero_mask.sum() >= 2:
    X_non_zero = X[non_zero_mask].toarray()
    indices_non_zero = valid_indices[non_zero_mask]

    actual_clusters = min(
        max(2, N_CLUSTERS),
        len(indices_non_zero),
    )

    clustering = AgglomerativeClustering(
        n_clusters=actual_clusters,
        metric="cosine",
        linkage="average",
    )

    labels = clustering.fit_predict(X_non_zero)
    motifs.loc[indices_non_zero, "cluster_id"] = labels

    if len(set(labels)) > 1 and len(labels) > len(set(labels)):
        silhouette = silhouette_score(X_non_zero, labels, metric="cosine")
        print(f"Score de silhouette : {silhouette:.3f}")
    else:
        silhouette = np.nan
else:
    silhouette = np.nan
    print("Pas assez de motifs exploitables pour créer des clusters.")

print("Le cluster -1 correspond aux motifs non exploitables ou non vectorisés.")
display(motifs[["reason_raw", "reason_model", "cluster_id"]].head(20))

In [ ]:
# Cette table est destinée à la revue humaine.
# Elle montre les expressions principales et des exemples dans chaque cluster.

feature_names = cluster_vectorizer.get_feature_names_out()
cluster_rows = []

for cluster_id in sorted(motifs.loc[motifs["cluster_id"] >= 0, "cluster_id"].unique()):
    cluster_indices = motifs.index[motifs["cluster_id"].eq(cluster_id)]
    cluster_text_positions = [
        np.where(valid_indices == index)[0][0]
        for index in cluster_indices
        if index in valid_indices
    ]

    cluster_text_positions = [
        pos for pos in cluster_text_positions
        if non_zero_mask[pos]
    ]

    if not cluster_text_positions:
        continue

    cluster_matrix = X[cluster_text_positions]
    mean_weights = np.asarray(cluster_matrix.mean(axis=0)).ravel()

    top_terms = [
        feature_names[position]
        for position in mean_weights.argsort()[::-1][:10]
        if mean_weights[position] > 0
    ]

    cluster_motifs = motifs.loc[cluster_indices].copy()

    examples = " | ".join(
        cluster_motifs["reason_raw"]
        .dropna()
        .head(5)
        .astype(str)
        .tolist()
    )

    cluster_rows.append({
        "cluster_id": cluster_id,
        "motifs_count": len(cluster_motifs),
        "top_ngrams": " ; ".join(top_terms),
        "median_gap_days": cluster_motifs["gap_days"].median(),
        "main_regions": " ; ".join(
            cluster_motifs[REGION_COL]
            .value_counts()
            .head(3)
            .index.astype(str)
            .tolist()
        ),
        "main_operators": " ; ".join(
            cluster_motifs[OPERATOR_COL]
            .value_counts()
            .head(3)
            .index.astype(str)
            .tolist()
        ),
        "examples": examples,
    })

cluster_summary = (
    pd.DataFrame(cluster_rows)
    .sort_values("motifs_count", ascending=False)
    .reset_index(drop=True)
)

display(cluster_summary)

In [ ]:
# Les clusters découvrent les formulations proches.
# La taxonomie métier donne ensuite des noms stables aux catégories.
#
# Les règles ci-dessous sont une première version, issue du notebook existant.
# Elles devront être ajustées après revue de cluster_summary.

THEME_TAXONOMY = [
    {
        "stable_theme": "signature_absente_ou_incomplete",
        "label": "Signature absente ou incomplète",
        "pattern": (
            r"signature_absente|signature_non_signee|"
            r"signature.*manquante|signataire"
        ),
    },
    {
        "stable_theme": "pvcl_constat_limites",
        "label": "PVCL et constats de limites",
        "pattern": r"\bpvcl\b|constat.*limite|limite.*plan",
    },
    {
        "stable_theme": "voisinage_limitrophes",
        "label": "Voisinage et limitrophes",
        "pattern": r"\bvoisin|voisine|limitrophe|famille|borne|amorce",
    },
    {
        "stable_theme": "plan_toponymie_noms",
        "label": "Plan, toponymie et noms",
        "pattern": r"\bplan\b|toponymie|nom.*complet|incoherence.*nom",
    },
    {
        "stable_theme": "pieces_documents_scan",
        "label": "Pièces et documents",
        "pattern": (
            r"document_a_fournir|document_a_joindre|piece_manquante|"
            r"\bdocument\b|\bscan|illisible|\bcni\b|contrat|declaration"
        ),
    },
    {
        "stable_theme": "audition_enquete_historique",
        "label": "Audition, enquête et historique",
        "pattern": r"audition|enquete|referent|cedant|demandeur|historique.*parcelle",
    },
    {
        "stable_theme": "droits_origine_transaction",
        "label": "Origine des droits et transaction",
        "pattern": r"origine.*droit|nature.*droit|transaction|succession|achat|don|cession",
    },
    {
        "stable_theme": "servitude_route_piste",
        "label": "Servitude, route et piste",
        "pattern": r"servitude|route|piste|emprise|chemin",
    },
    {
        "stable_theme": "shapefile_geometrie_superficie",
        "label": "Shapefile, géométrie et superficie",
        "pattern": r"shapefile|superficie|coordonne|geometr|polygon|parcelle",
    },
    {
        "stable_theme": "qualite_information_insuffisante",
        "label": "Information insuffisante ou à compléter",
        "pattern": (
            r"information_a_completer|information_a_preciser|"
            r"information_a_renseigner|information_manquante|"
            r"champ_non_renseigne|incomplet|insuffisant"
        ),
    },
]

# Après analyse de cluster_summary, renseigner ici les associations validées.
# Exemple : 4: "signature_absente_ou_incomplete"
# Garder ce dictionnaire vide lors du premier lancement.
CLUSTER_THEME_MAPPING = {
    # 0: "signature_absente_ou_incomplete",
    # 1: "pvcl_constat_limites",
}


def assign_theme_by_rule(text):
    """Attribue le premier thème dont la règle correspond au motif."""
    text = str(text)

    for theme in THEME_TAXONOMY:
        if re.search(theme["pattern"], text, flags=re.IGNORECASE):
            return theme["stable_theme"]

    return None


def assign_stable_theme(row):
    """
    Priorité :
    1. règle métier directe ;
    2. association manuelle validée entre cluster et thème ;
    3. autre_a_revoir.
    """
    rule_theme = assign_theme_by_rule(row["reason_model"])

    if rule_theme:
        return pd.Series({
            "stable_theme": rule_theme,
            "theme_assignment_method": "rule",
            "needs_review": False,
        })

    if row["cluster_id"] in CLUSTER_THEME_MAPPING:
        return pd.Series({
            "stable_theme": CLUSTER_THEME_MAPPING[row["cluster_id"]],
            "theme_assignment_method": "cluster_mapping",
            "needs_review": False,
        })

    return pd.Series({
        "stable_theme": "autre_a_revoir",
        "theme_assignment_method": "manual_review",
        "needs_review": True,
    })


motifs[["stable_theme", "theme_assignment_method", "needs_review"]] = motifs.apply(
    assign_stable_theme,
    axis=1,
)

taxonomy_df = pd.DataFrame(THEME_TAXONOMY)
taxonomy_df = pd.concat([
    taxonomy_df,
    pd.DataFrame([{
        "stable_theme": "autre_a_revoir",
        "label": "Autres motifs à revoir",
        "pattern": "Aucune règle ou correspondance de cluster validée",
    }]),
], ignore_index=True)

display(motifs[[
    "reason_raw",
    "reason_model",
    "cluster_id",
    "stable_theme",
    "theme_assignment_method",
    "needs_review",
]].head(20))

In [ ]:
# Cette étape mesure la couverture réelle de la taxonomie.
# Les motifs à revoir guideront les futures améliorations des règles.

theme_summary = (
    motifs.groupby("stable_theme", dropna=False)
    .agg(
        motifs_count=("reason_id", "count"),
        events_count=("event_id", "nunique"),
        median_gap_days=("gap_days", "median"),
        p90_gap_days=("gap_days", lambda values: values.quantile(0.90)),
        regions_count=(REGION_COL, "nunique"),
        operators_count=(OPERATOR_COL, "nunique"),
    )
    .reset_index()
)

theme_summary["motifs_pct"] = (
    theme_summary["motifs_count"] / len(motifs) * 100
).round(2)

theme_summary = theme_summary.sort_values(
    "motifs_count",
    ascending=False,
).reset_index(drop=True)

coverage_pct = (
    100
    - theme_summary.loc[
        theme_summary["stable_theme"].eq("autre_a_revoir"),
        "motifs_pct"
    ].sum()
)

motifs_to_review = motifs[
    motifs["needs_review"]
].sort_values(
    ["cluster_id", "reason_word_count"],
    ascending=[True, False],
)

print(f"Couverture de la taxonomie : {coverage_pct:.1f}%")
print(f"Motifs à revoir : {len(motifs_to_review):,}")

display(theme_summary)
display(motifs_to_review.head(50))

In [ ]:
# Les tableaux suivants servent directement à l'analyse métier.

themes_by_region = pd.crosstab(
    motifs[REGION_COL],
    motifs["stable_theme"],
    normalize="index",
).mul(100).round(2)

themes_by_operator = pd.crosstab(
    motifs[OPERATOR_COL],
    motifs["stable_theme"],
    normalize="index",
).mul(100).round(2)

delay_by_region_operator = (
    motifs.groupby([REGION_COL, OPERATOR_COL])
    .agg(
        motifs_count=("reason_id", "count"),
        median_gap_days=("gap_days", "median"),
        p90_gap_days=("gap_days", lambda values: values.quantile(0.90)),
    )
    .reset_index()
    .sort_values("motifs_count", ascending=False)
)

motifs["rejected_month"] = (
    pd.to_datetime(motifs["rejected_at"], errors="coerce")
    .dt.to_period("M")
    .astype("string")
)

monthly_evolution = (
    motifs.groupby(["rejected_month", "stable_theme"], dropna=False)
    .agg(
        motifs_count=("reason_id", "count"),
        median_gap_days=("gap_days", "median"),
    )
    .reset_index()
    .sort_values(["rejected_month", "motifs_count"])
)

display(delay_by_region_operator.head(20))
display(themes_by_region)
display(themes_by_operator)
display(monthly_evolution.head(30))

In [ ]:
# Les graphiques sont affichés dans le notebook et enregistrés dans figures/.
# Ils seront aussi insérés dans le rapport Excel.

theme_order = theme_summary["stable_theme"].tolist()

plt.figure(figsize=(12, 7))
sns.barplot(
    data=theme_summary,
    x="motifs_count",
    y="stable_theme",
    order=theme_order,
    color="#4c78a8",
)
plt.title("Volume de motifs par thème")
plt.xlabel("Nombre de motifs")
plt.ylabel("Thème")
plt.tight_layout()
theme_volume_chart = FIGURES_DIR / "volume_par_theme.png"
plt.savefig(theme_volume_chart, dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(12, 7))
sns.boxplot(
    data=motifs[motifs["stable_theme"].isin(theme_order)],
    x="gap_days",
    y="stable_theme",
    order=theme_order,
    color="#fdbb84",
)
plt.title("Délais de correction par thème")
plt.xlabel("Délai de correction (jours)")
plt.ylabel("Thème")
plt.tight_layout()
theme_delay_chart = FIGURES_DIR / "delais_par_theme.png"
plt.savefig(theme_delay_chart, dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(14, 7))
sns.heatmap(
    themes_by_region,
    cmap="YlGnBu",
    linewidths=0.4,
    linecolor="white",
)
plt.title("Part des thèmes par région (%)")
plt.xlabel("Thème")
plt.ylabel("Région")
plt.tight_layout()
region_heatmap_chart = FIGURES_DIR / "themes_par_region.png"
plt.savefig(region_heatmap_chart, dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(14, 6))
sns.heatmap(
    themes_by_operator,
    cmap="YlOrRd",
    linewidths=0.4,
    linecolor="white",
)
plt.title("Part des thèmes par opérateur (%)")
plt.xlabel("Thème")
plt.ylabel("Opérateur")
plt.tight_layout()
operator_heatmap_chart = FIGURES_DIR / "themes_par_operateur.png"
plt.savefig(operator_heatmap_chart, dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# Cette étape crée le document final.
# Il contient les résultats détaillés, les contrôles qualité et les graphiques.

resume_executif = pd.DataFrame({
    "indicateur": [
        "Lignes source",
        "Événements extraits",
        "Motifs atomiques",
        "Motifs valides",
        "Motifs à revoir",
        "Couverture de la taxonomie (%)",
        "Nombre de clusters",
        "Score de silhouette",
    ],
    "valeur": [
        len(df),
        len(events),
        len(motifs),
        int(motifs["is_valid_reason"].sum()),
        len(motifs_to_review),
        round(coverage_pct, 2),
        int(motifs["cluster_id"].nunique() - (1 if -1 in motifs["cluster_id"].unique() else 0)),
        round(silhouette, 3) if pd.notna(silhouette) else "Non calculé",
    ],
})

# Export CSVs utiles pour de futurs traitements ou pour ADAC.
events.to_csv(EVENTS_PATH, index=False, encoding="utf-8-sig")
motifs.to_csv(MOTIFS_PATH, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(REPORT_PATH, engine="openpyxl") as writer:
    resume_executif.to_excel(writer, sheet_name="resume_executif", index=False)
    quality_data.to_excel(writer, sheet_name="qualite_donnees", index=False)
    motifs.to_excel(writer, sheet_name="motifs_details", index=False)
    theme_summary.to_excel(writer, sheet_name="themes_resume", index=False)
    cluster_summary.to_excel(writer, sheet_name="clusters", index=False)
    themes_by_region.reset_index().to_excel(
        writer,
        sheet_name="themes_par_region",
        index=False,
    )
    themes_by_operator.reset_index().to_excel(
        writer,
        sheet_name="themes_par_operateur",
        index=False,
    )
    delay_by_region_operator.to_excel(
        writer,
        sheet_name="delais_region_operateur",
        index=False,
    )
    monthly_evolution.to_excel(
        writer,
        sheet_name="evolution_mensuelle",
        index=False,
    )
    motifs_to_review.to_excel(
        writer,
        sheet_name="motifs_a_revoir",
        index=False,
    )
    taxonomy_df.to_excel(writer, sheet_name="taxonomie", index=False)
    ngram_counts.head(200).to_excel(
        writer,
        sheet_name="ngrams_frequents",
        index=False,
    )

# Ajout des graphiques dans une feuille dédiée.
workbook = load_workbook(REPORT_PATH)
worksheet = workbook.create_sheet("graphiques")

chart_positions = [
    (theme_volume_chart, "A1"),
    (theme_delay_chart, "A28"),
    (region_heatmap_chart, "J1"),
    (operator_heatmap_chart, "J28"),
]

for image_path, anchor in chart_positions:
    if image_path.exists():
        image = ExcelImage(str(image_path))
        image.width = 650
        image.height = 360
        worksheet.add_image(image, anchor)

# Mise en forme minimale : filtre, ligne d'en-tête figée et largeur adaptée.
for sheet_name in workbook.sheetnames:
    ws = workbook[sheet_name]

    if ws.max_row > 1:
        ws.auto_filter.ref = ws.dimensions
        ws.freeze_panes = "A2"

    for column_cells in ws.columns:
        column_letter = column_cells[0].column_letter
        max_length = max(
            len(str(cell.value)) if cell.value is not None else 0
            for cell in column_cells[:1000]
        )
        ws.column_dimensions[column_letter].width = min(max(max_length + 2, 12), 45)

workbook.save(REPORT_PATH)

print(f"Rapport Excel créé : {REPORT_PATH}")
print(f"Événements exportés : {EVENTS_PATH}")
print(f"Motifs détaillés exportés : {MOTIFS_PATH}")

In [ ]:
# Cette dernière cellule produit une synthèse rapide à lire après chaque exécution.

top_theme = theme_summary.iloc[0]
slowest_theme = (
    theme_summary
    .dropna(subset=["median_gap_days"])
    .sort_values("median_gap_days", ascending=False)
    .iloc[0]
)

print("SYNTHÈSE AUTOMATIQUE")
print("-" * 60)
print(
    f"Le thème le plus fréquent est "
    f"« {top_theme['stable_theme']} » "
    f"avec {top_theme['motifs_count']:,} motifs "
    f"({top_theme['motifs_pct']:.1f}%)."
)
print(
    f"Le thème au délai médian le plus élevé est "
    f"« {slowest_theme['stable_theme']} » "
    f"avec {slowest_theme['median_gap_days']:.1f} jours."
)
print(f"La couverture actuelle de la taxonomie est de {coverage_pct:.1f}%.")
print(
    f"{len(motifs_to_review):,} motifs sont dans "
    "« autre_a_revoir » et doivent être examinés pour améliorer les règles."
)

In [3]:
print("Dossier courant :", Path.cwd())

NameError: name 'Path' is not defined